In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
from matplotlib import pyplot as plt
%matplotlib inline

In [2]:
df=pd.read_csv('insurance_data.csv')

In [3]:
df.head()

,age,affordibility,bought_insurance
0,22,1,0
1,25,0,0
2,47,1,1
3,52,0,0
4,46,1,1


In [4]:
from sklearn.model_selection import train_test_split
X=df[['age','affordibility']]
y=df['bought_insurance']
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [5]:
X_train_scaled=X_train.copy()
X_train_scaled['age']=X_train_scaled['age']/100
X_test_scaled=X_test.copy()
X_test_scaled['age']=X_test_scaled['age']/100


In [6]:
X_train_scaled

,age,affordibility
17,0.58,1
22,0.40,1
11,0.28,1
13,0.29,0
15,0.55,1
1,0.25,0
4,0.46,1
5,0.56,1
2,0.47,1
16,0.25,0


In [7]:
def sigmoid(x):
    return 1/(1+np.exp(-x))

In [8]:
def log_loss(y_true,y_predicted):
    epsilon = 1e-15
    y_predicted_final = [max(i,epsilon) for i in y_predicted]
    y_predicted_final = [min(i,1-epsilon) for i in y_predicted_final]
    #successfully avoided log(0) and log(1)
    y_predicted_final = np.array(y_predicted_final)
    return -np.mean(y_true * np.log(y_predicted_final) + (1-y_true) * np.log(1-y_predicted_final))

In [24]:
w1 = w2 = 1 
bias = 0 

In [25]:
def gradient_descent(age, affordibility, y_true , epochs):
    global w1, w2, bias
    rate = 0.5
    n=len(age)
    for i in range (epochs):
        y = (w1*age) + (w2*affordibility) + bias 
        y_predicted = sigmoid(y)
        loss = log_loss(y_true,y_predicted)
        #derivatives 
        w1d = (1/n)*np.dot(np.transpose(age),(y_predicted - y_true))
        w2d = (1/n)*np.dot(np.transpose(affordibility),(y_predicted - y_true))
        bd = np.mean(y_predicted-y_true)
        #adjustments
        w1 = w1 - rate*w1d
        w2 = w2 - rate*w2d
        bias = bias - rate*bd
        if(epochs%50==0):
            print(f'Epoch:{i} , w1:{w1} , w2:{w2} , bias:{bias} , loss: {loss}')

In [26]:
def fit(X, y, epochs):
        return gradient_descent(X['age'], X['affordibility'], y, epochs)

In [27]:
def predict(X):
    y = w1*X['age'] + w2*X['affordibility'] + bias
    return sigmoid(y)

In [34]:
y_test

9     1
25    1
8     1
21    0
0     0
12    0
Name: bought_insurance, dtype: int64

In [30]:
fit(X_train_scaled, y_train, 501)

In [31]:
y_pred = predict(X_test_scaled)
print(y_pred[:5])

9     0.904226
25    0.839773
8     0.911264
21    0.099825
0     0.262319
dtype: float64


Or we can just make a class

In [ ]:
class CustomModel:
    def __init__(self):
        self.w1 = 1
        self.w2 = 1
        self.bias = 0
    
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))
    
    def log_loss(self, y_true, y_predicted):
        epsilon = 1e-15
        #clipping auto this time 
        y_predicted = np.clip(y_predicted, epsilon, 1 - epsilon)
        return -np.mean(y_true*np.log(y_predicted) + (1-y_true)*np.log(1-y_predicted))
    
    def gradient_descent(self, age, affordibility, y_true, epochs):
        n = len(age)
        rate = 0.5

        for i in range(epochs):
            y_linear = (self.w1*age) + (self.w2*affordibility) + self.bias
            y_predicted = self.sigmoid(y_linear)

            loss = self.log_loss(y_true, y_predicted)

            # gradients
            w1d = (1/n) * np.dot(age.T, (y_predicted - y_true))
            w2d = (1/n) * np.dot(affordibility.T, (y_predicted - y_true))
            bd  = np.mean(y_predicted - y_true)

            # updates
            self.w1 -= rate * w1d
            self.w2 -= rate * w2d
            self.bias -= rate * bd

            if i % 50 == 0:
                print(f"Epoch {i}: loss={loss:.4f}, w1={self.w1:.3f}, w2={self.w2:.3f}, bias={self.bias:.3f}")

    def fit(self, X, y, epochs=1000):
        self.gradient_descent(X['age'], X['affordibility'], y, epochs)
    
    def predict(self, X):
        linear = self.w1*X['age'] + self.w2*X['affordibility'] + self.bias
        return self.sigmoid(linear)

In [ ]:
model = CustomModel()
model.fit(X_train_scaled, y_train, epochs=501)

In [ ]:
y_pred = model.predict(X_test_scaled)
print(y_pred[:5])